# Visign — Colab Training Notebook

Run the cells **in order**. Each section is independent once setup (§1) and feature upload (§2) are done.

**Before you start:** `Runtime → Change runtime type → GPU (T4)`.

Reference doc with explanations of every flag: [`docs/COLAB_TRAINING_GUIDE.md`](https://github.com/trandongtruclam/visign-ai-model/blob/main/docs/COLAB_TRAINING_GUIDE.md)

## §1. One-time Colab session setup
Mount Drive, clone repo, install deps, confirm GPU.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content
!git clone https://github.com/trandongtruclam/visign-ai-model.git || (cd visign-ai-model && git pull)
%cd visign-ai-model

In [ ]:
!pip install -q torch numpy pandas scikit-learn tqdm scipy huggingface_hub

In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("!! WARNING: No GPU. Switch runtime to GPU (T4) before training.")

## §2. Upload preprocessed features to Colab

The repo does **not** ship `preprocessed_npz/` (14k files, gitignored) or `index.csv`.

**Run once on your local Windows machine** (in `D:/hub/sudocode/visign/ai-model`):

```powershell
tar -czf visign_features.tar.gz preprocessed_npz index.csv splits.json
```

Upload `visign_features.tar.gz` to Drive at `My Drive/visign/visign_features.tar.gz`.

⚠️ **Do not use `Compress-Archive` (.zip)** — it mangles Vietnamese filenames. Use `tar` only.

In [ ]:
%cd /content/visign-ai-model
!tar -xzf "/content/drive/MyDrive/visign/visign_features.tar.gz" -C /content/visign-ai-model

In [ ]:
import os, glob
print("cwd:", os.getcwd())
print("index.csv:", os.path.isfile("index.csv"))
print("preprocessed_npz/:", os.path.isdir("preprocessed_npz"))
print("npy count:", len(glob.glob("preprocessed_npz/*.npy")))
print("first 3:", sorted(glob.glob("preprocessed_npz/*.npy"))[:3])
print()
print("If the filenames above show mojibake (e.g. 'sс╗С kh├┤ng') instead of")
print("Vietnamese characters, your tarball was corrupted on upload — re-create")
print("it with `tar -czf` (NOT Compress-Archive) and re-upload.")

## §3. Train the model

Run the BiLSTM + attention training loop. Saves `artifacts/best_model.pt` whenever val F1 improves; early-stops if it stops improving for 8 epochs.

**Healthy first-epoch output:**
```
Using device: cuda
Loaded ~14000 samples across 274 classes
Train samples: ~12200, Val samples: ~850, Test samples (held out, untouched): ~900
Persisted split manifest to artifacts/splits.json
Input dimension detected: 628
Epoch 001 | train_loss=... val_loss=... train_f1=... val_f1=... lr=0.001000
```

The `Test samples (held out, untouched)` line is your confirmation that the source-level split is in effect.

In [ ]:
%cd /content/visign-ai-model
!python src/train/modeling.py \
    --index-csv index.csv \
    --feature-dir preprocessed_npz \
    --output-dir artifacts \
    --epochs 60 --batch-size 32 --lr 1e-3 \
    --use-class-weights --label-smoothing 0.05 \
    --num-workers 2 --device cuda

## §4. Persist artifacts back to Drive

Colab disconnects after ~90 min idle / ~12 h wall. Run this whenever you want a checkpoint preserved (safe to re-run; just overwrites).

In [ ]:
import shutil, os
drive_dir = "/content/drive/MyDrive/visign/artifacts"
os.makedirs(drive_dir, exist_ok=True)
for f in ["best_model.pt", "training_history.json", "splits.json"]:
    src = f"/content/visign-ai-model/artifacts/{f}"
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_dir, f))
        print("copied", f)
    else:
        print("missing", f)

## §5. Evaluate on the held-out test split

Writes `docs/EVAL_REPORT.md`, `docs/eval_summary.json`, `docs/eval_per_class.csv`, `docs/eval_confusion_matrix.npy`.

In [ ]:
%cd /content/visign-ai-model
!python src/eval/evaluate.py \
    --checkpoint artifacts/best_model.pt \
    --index-csv index.csv \
    --feature-dir preprocessed_npz \
    --split test \
    --output-dir docs \
    --top-k 5 \
    --device cuda \
    --skip-latency

In [ ]:
from IPython.display import Markdown, display
with open("/content/visign-ai-model/docs/EVAL_REPORT.md", "r", encoding="utf-8") as f:
    display(Markdown(f.read()))

---

## §6. (Optional) Expand val/test with VOYA_VSL

**Run §6 only if your test set is too small to be meaningful** (e.g. < 100 samples). It downloads ~3 GB of public VOYA_VSL keypoints and uses them as additional val/test sources, growing your eval set from ~35 samples to ~6,000.

Safe to skip on subsequent training runs once VOYA is already imported (it's recorded in `splits.json` and `index.csv`).

### §6.1 Sanity-check: how many of your classes overlap with VOYA?

In [ ]:
import json, urllib.request
voya = json.loads(urllib.request.urlopen(
    "https://huggingface.co/datasets/Kateht/VOYA_VSL/resolve/main/labels.json").read())
voya_names = {v.strip().lower() for v in voya.values()}
mine = {k.split("/", 1)[0].split(" _")[0].strip().lower()
        for k in json.load(open("/content/visign-ai-model/splits.json"))["sources"]}
overlap = mine & voya_names
print(f"exact overlap: {len(overlap)} / {len(mine)} of your classes")
print("\nDecision rule:")
print("  >= 80 classes → run §6.2 (worth the effort)")
print("  30–80 classes → marginal but probably worth it")
print("  <  30 classes → skip §6 entirely")

### §6.2 Pull VOYA, compute features, append to `index.csv` + `splits.json`

The script does the full integration end-to-end — **no rerun of `preprocess_pipeline.py` is needed**:

- Downloads each overlapping VOYA class file (~340 MB each, ~50 GB total) one-by-one. Source files are deleted right after sampling, so peak disk stays under 1 GB.
- For each chosen clip: converts VOYA's `(60, 1605)` keypoints into our 25-pose-landmark layout, runs the **exact same** `preprocess_sample` the trainer expects (so VOYA features are 628-D and indistinguishable from QIPEDC features), writes `preprocessed_npz/sample_<row>_<label>.npy`, and appends a row to `index.csv`.
- Marks each new source as `val` or `test` in `splits.json`.

**Re-run safe:** sources already in `index.csv` are skipped, so re-running won't duplicate samples.

~30 % of class IDs 404 (VOYA's `labels.json` lists more entries than actual files). Auto-skipped. ~15 min on a fast Colab connection.

In [ ]:
%cd /content/visign-ai-model
!python src/keypoints/voya_import.py \
    --splits-json splits.json \
    --index-csv index.csv \
    --feature-dir preprocessed_npz \
    --n-val 20 --n-test 20 \
    --seed 42

### §6.3 ~~Rebuild `index.csv`~~ — skip this section

`voya_import.py` now appends rows to `index.csv` and writes `.npy` features directly. **Do not** rerun `preprocess_pipeline.py` here — it would rebuild `index.csv` from `augmented/`, which on Colab only contains the new VOYA dumps (so you'd wipe out all 14k QIPEDC rows).

The next cell is intentionally a no-op placeholder; you can skip it.

In [ ]:
# §6.3 placeholder — do nothing.
# voya_import.py already updated index.csv and preprocessed_npz/ in §6.2.
print("skipped: voya_import.py already produced index.csv + .npy features end-to-end")

### §6.4 Sanity-check the new splits

**Expected after VOYA import:**
```
rows: ~20000
split
train    ~14000
val       ~3000
test      ~3000
```

- `train` should still be ~14000 (all your original QIPEDC rows untouched).
- `val` and `test` should each have grown to a few thousand.

If `train` is in the hundreds, something rebuilt `index.csv` from `augmented/` and dropped the QIPEDC rows. Recovery: re-extract the original `visign_features.tar.gz` from Drive and rerun §6.2.

In [ ]:
import pandas as pd
df = pd.read_csv("/content/visign-ai-model/index.csv")
print("rows:", len(df))
print(df["split"].value_counts())
print("\nclasses per split:")
print(df.groupby("split")["label"].nunique())

### §6.5 Retrain + re-evaluate on the bigger test set

Same commands as §3 and §5, but now the metrics will actually be meaningful.

In [ ]:
%cd /content/visign-ai-model
!python src/train/modeling.py \
    --index-csv index.csv \
    --feature-dir preprocessed_npz \
    --output-dir artifacts \
    --epochs 60 --batch-size 32 --lr 1e-3 \
    --use-class-weights --label-smoothing 0.05 \
    --num-workers 2 --device cuda

In [ ]:
%cd /content/visign-ai-model
!python src/eval/evaluate.py \
    --checkpoint artifacts/best_model.pt \
    --index-csv index.csv \
    --feature-dir preprocessed_npz \
    --split test \
    --output-dir docs \
    --top-k 5 \
    --device cuda \
    --skip-latency

In [ ]:
from IPython.display import Markdown, display
with open("/content/visign-ai-model/docs/EVAL_REPORT.md", "r", encoding="utf-8") as f:
    display(Markdown(f.read()))

### §6.6 Persist VOYA-augmented results back to Drive

In [ ]:
import shutil, os
drive_dir = "/content/drive/MyDrive/visign/artifacts_voya"
os.makedirs(drive_dir, exist_ok=True)
for f in ["best_model.pt", "training_history.json", "splits.json"]:
    src = f"/content/visign-ai-model/artifacts/{f}"
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_dir, f))
        print("copied", f)
# Also back up the EVAL_REPORT and per-class numbers
for f in ["EVAL_REPORT.md", "eval_summary.json", "eval_per_class.csv"]:
    src = f"/content/visign-ai-model/docs/{f}"
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_dir, f))
        print("copied", f)